It loads a checkpoint from `logs/` and opens an interactive MuJoCo viewer

- **A** = blind, **B** = depth (terrain only), **C** = depth (terrain + feet)

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import glob, time
import numpy as np
import jax
import mujoco, mujoco.viewer
import mediapy as media
from brax.training.acme import running_statistics
from brax.training.agents.ppo import checkpoint as ppo_checkpoint
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import networks_vision as ppo_networks_vision
from mujoco_playground import registry, wrapper
from mujoco_playground.config import locomotion_params

In [ ]:
FLAT_OV = {"goal_config.dist_range": [1.0, 5.0], "goal_config.segment_time": 6.0}

RUNS = {
    "flat_A":   ("flat5m_a_heading", "Go2GoalFlatTerrain", None),
    "flat_B":   ("flat5m_b_heading", "Go2GoalFlatTerrain", [0]),
    "flat_C":   ("flat5m_c_heading", "Go2GoalFlatTerrain", [0, 3]),
    "bridge_A": ("bridge_a_heading", "Go2GoalBridge",      None),
    "bridge_B": ("bridge_b_heading", "Go2GoalBridge",      [0]),
    "bridge_C": ("bridge_c_heading", "Go2GoalBridge",      [0, 3]),
    "beams_A":  ("beams_a_heading",  "Go2GoalBeams",       None),
    "beams_B":  ("beams_b_heading",  "Go2GoalBeams",       [0]),
    "beams_C":  ("beams_c_heading",  "Go2GoalBeams",       [0, 3]),
}

In [ ]:
def find_step_dir(path, step=None):
    path = os.path.abspath(path)
    if os.path.isdir(os.path.join(path, "checkpoints")):
        path = os.path.join(path, "checkpoints")
    steps = sorted((d for d in os.listdir(path) if d.isdigit()), key=int)
    if not steps:
        return path
    if step is None:
        return os.path.join(path, steps[-1])
    return os.path.join(path, min(steps, key=lambda s: abs(int(s) - step)))


def load(name, step=None):
    run, env_name, groups = RUNS[name]
    over = {"impl": "warp"}
    if env_name == "Go2GoalFlatTerrain":
        over.update(FLAT_OV)                      # same settings as in training
    vision = groups is not None
    if vision:
        over.update({"vision": True, "vision_config.nworld": 1,
                     "vision_config.enabled_geom_groups": list(groups)})
    env = registry.load(env_name, config=registry.get_default_config(env_name), config_overrides=over)
    rl = (locomotion_params.brax_vision_ppo_config if vision
          else locomotion_params.brax_ppo_config)(env_name, "warp")
    norm = running_statistics.normalize if rl.normalize_observations else (lambda x, y: x)
    make = ppo_networks_vision.make_ppo_networks_vision if vision else ppo_networks.make_ppo_networks
    net = make(env.observation_size, env.action_size, preprocess_observations_fn=norm, **rl.network_factory)
    folder = sorted(glob.glob(f"logs/*-{run}"))[-1]
    ckpt = find_step_dir(folder, step)
    print("checkpoint:", ckpt)
    policy = ppo_networks.make_inference_fn(net)(ppo_checkpoint.load(ckpt), deterministic=True)
    return env, policy


def _sphere(geom, xy):
    mujoco.mjv_initGeom(geom, mujoco.mjtGeom.mjGEOM_SPHERE, np.array([0.06, 0, 0]),
                        np.array([float(xy[0]), float(xy[1]), 0.06]), np.eye(3).flatten(),
                        np.array([1, 0, 0, 1], np.float32))


def _upscale(depth, h=240):
    # depth image -> grayscale RGB, rotated so that forward points up
    g = (np.clip(np.rot90(depth, 1), 0, 1) * 255).astype(np.uint8)
    ys = np.arange(h) * g.shape[0] // h
    w = h * g.shape[1] // g.shape[0]
    xs = np.arange(w) * g.shape[1] // w
    return np.ascontiguousarray(np.repeat(g[ys][:, xs][..., None], 3, axis=-1))


def _follow_cam(cam, m):
    cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
    cam.trackbodyid = m.body("base").id
    cam.distance, cam.elevation, cam.azimuth = 3.0, -30.0, 135.0


def _setup(env, policy, seed):
    wenv = wrapper.wrap_for_brax_training(env, episode_length=10**6, action_repeat=1)
    reset, step = jax.jit(wenv.reset), jax.jit(wenv.step)
    infer = jax.jit(jax.vmap(policy))
    key = jax.random.PRNGKey(seed)
    key, k = jax.random.split(key)
    return reset, step, infer, key, reset(jax.random.split(k, 1))


def _new_target(env, state, key):
    xy = env._sample_target(key, state.data.qpos[0, :2])
    state.info["target_xy"] = state.info["target_xy"].at[0].set(xy)
    state.info["steps_left"] = state.info["steps_left"].at[0].set(env._segment_steps)
    return state


def _depth(state):
    if isinstance(state.obs, dict) and "pixels/depth" in state.obs:
        return np.asarray(state.obs["pixels/depth"][0, ..., 0])
    return None


def play(env, policy, seed=0, speed=1.0):
    # interactive viewer: N = new target, K = reset, ESC = exit
    reset, step, infer, key, state = _setup(env, policy, seed)
    pressed = {"N": False, "K": False}

    def on_key(code):
        c = chr(code) if 32 <= code < 128 else ""
        if c in pressed:
            pressed[c] = True

    m, d = env.mj_model, mujoco.MjData(env.mj_model)
    with mujoco.viewer.launch_passive(m, d, key_callback=on_key) as v:
        _follow_cam(v.cam, m)
        while v.is_running():
            t0 = time.time()
            if pressed["K"]:
                pressed["K"] = False
                key, k = jax.random.split(key)
                state = reset(jax.random.split(k, 1))
            if pressed["N"]:
                pressed["N"] = False
                key, k = jax.random.split(key)
                state = _new_target(env, state, k)
            key, k = jax.random.split(key)
            act, _ = infer(state.obs, jax.random.split(k, 1))
            state = step(state, act)
            with v.lock():
                d.qpos[:] = np.asarray(state.data.qpos[0])
                d.qvel[:] = np.asarray(state.data.qvel[0])
                mujoco.mj_forward(m, d)
                v.user_scn.ngeom = 0
                _sphere(v.user_scn.geoms[0], np.asarray(state.info["target_xy"][0]))
                v.user_scn.ngeom = 1
            dep = _depth(state)
            if dep is not None:
                try:
                    img = _upscale(dep)
                    v.set_images((mujoco.MjrRect(10, 10, img.shape[1], img.shape[0]), img))
                except Exception:
                    pass
            v.sync()
            time.sleep(max(0.0, env.dt / speed - (time.time() - t0)))


def record(env, policy, seed=0, targets=1, h=360, w=480):
    # Returns video frames. Left: robot; right: depth image (B and C only).
    reset, step, infer, key, state = _setup(env, policy, seed)
    m, d = env.mj_model, mujoco.MjData(env.mj_model)
    r = mujoco.Renderer(m, height=h, width=w)
    cam = mujoco.MjvCamera()
    _follow_cam(cam, m)
    frames = []
    for seg in range(targets):
        if seg > 0:
            key, k = jax.random.split(key)
            state = _new_target(env, state, k)
        for _ in range(env._segment_steps - 1):
            key, k = jax.random.split(key)
            act, _ = infer(state.obs, jax.random.split(k, 1))
            state = step(state, act)
            d.qpos[:] = np.asarray(state.data.qpos[0])
            d.qvel[:] = np.asarray(state.data.qvel[0])
            mujoco.mj_forward(m, d)
            r.update_scene(d, camera=cam)
            if r.scene.ngeom < r.scene.maxgeom:
                _sphere(r.scene.geoms[r.scene.ngeom], np.asarray(state.info["target_xy"][0]))
                r.scene.ngeom += 1
            frame = r.render()
            dep = _depth(state)
            if dep is not None:
                frame = np.concatenate([frame, _upscale(dep, h)], axis=1)
            frames.append(frame)
    r.close()
    return frames

Viewer, change `NAME` to any key of `RUNS`.

In [ ]:
NAME = "bridge_C"  #"beams_A"
env, pol = load(NAME)
play(env, pol, seed=0)

In [ ]:
# frames = record(env, pol, seed=0)
# media.show_video(frames, fps=round(1 / env.dt))